# 06 — Chat Formatting Token Tax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/06_chat_formatting_token_tax.ipynb)

Compare plain text, code, JSON, and chat-style formatting with the same local tokenizer.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
%pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
%pip install -q -e .

import sys
src = '/content/forge-tokenizer/src'
if src not in sys.path:
    sys.path.insert(0, src)
print('Ready. Run the Imports cell next.')

## 2. Imports

In [ ]:
from pathlib import Path
assert Path('src/forge_tokenizer').exists(), 'Run section 1: Install & Clone first.'
import json
import os
os.environ.setdefault('MPLCONFIGDIR', 'generated/.mplconfig')
Path('generated/.mplconfig').mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt

from forge_tokenizer import ForgeTokenizer
from forge_tokenizer.metrics import formatting_overhead

print('forge-tokenizer ready')

## 3. Train The Local Tokenizer

In [ ]:
corpus = Path('data/tiny_corpus.txt').read_text(encoding='utf-8')
tokenizer = ForgeTokenizer().train(corpus, num_merges=100)
print('vocab size:', tokenizer.vocab_size)

## 4. Same Intent, Different Formatting

In [ ]:
plain = 'Explain tokenization in one short sentence.'
code = "def explain():\n    return 'Tokenization turns text into token ids.'\n"
payload = {'task': 'explain', 'topic': 'tokenization', 'length': 'short'}
json_text = json.dumps(payload, indent=2)
chat = "<system>\nYou are concise.\n</system>\n<user>\nExplain tokenization in one short sentence.\n</user>\n<assistant>"

variants = {
    'plain': plain,
    'code': code,
    'json': json_text,
    'chat': chat,
}

for name, text in variants.items():
    print('\n---', name, '---')
    print(text)
    print('tokens:', len(tokenizer.encode(text)))

## 5. Plot Formatting Overhead

In [ ]:
baseline = len(tokenizer.encode(plain))
labels = list(variants)
counts = [len(tokenizer.encode(variants[label])) for label in labels]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars = ax.bar(labels, counts, color='#7c3aed', edgecolor='#111827', linewidth=0.8)
ax.axhline(baseline, color='#111827', linewidth=1, linestyle='--', label='plain baseline')
ax.set_title('Formatting changes token count')
ax.set_ylabel('Tokens')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='y', alpha=0.22)
for bar, value in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, value, str(value), ha='center', va='bottom', fontsize=9)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 6. Code Formatting Overhead

In [ ]:
code_sample = Path('data/code_sample.py.txt').read_text(encoding='utf-8')
print(code_sample)
print('formatting overhead:', formatting_overhead(tokenizer, code_sample))

---

You now have the simple path: text → tokens → metrics → embeddings → logits → formatted prompts.